# 30 - Refining the team encoder (experiment 14)

The team features are the only clear gain so far (+0.90 points). The final standings use the other 70% of the test data,
so the teams of the final set relate to the training teams as those of the test set do. P1 teams never seen: 26.9% in
the test set, 29.2% in CV validation folds, about 52% in the training rows under the current mask (`train_unseen=0.3`).
A descriptive check on the training battles also shows that teams do differ: the win rates of two random halves of a team's
battles correlate 0.43 for teams with 10-19 battles (49 teams) and 0.48 for 100 or more (7 teams), on small samples. Items,
levels and slot order add nothing to the identity of a team.

Three candidates against R1 + R3 (submissions 27-29), measured once, with no further iteration:

- **No mask (`pairs=True`, `train_unseen=0`):** the mask marks about twice as many rows as unknown as the test set has. In
  experiment 11 it changed little on clean CV (+0.90 without, +0.88 with), so the expected gain is small;
- **B, `soft_p2=True` (idea R2):** 55.6% of the battles enter the pool through a fully revealed P2 team. P2 teams
  revealed in part (3 to 5 species) now enter too, with their result spread over the compatible teams;
- **C, `residual=True`:** the pool counts results above what a small logistic regression expects from the state at turn 30,
  to make the rates less noisy and less redundant with the other features.

Dropped before running: `n_inner=20`, which increases the anticorrelation inside a team, and the combinations.

**Decision rule (fixed before running)**, LightGBM of experiment 04, paired on 15 folds (seeds 42 / 7 / 123 × 5), against
R1 + R3. For the candidate with the best clean CV:
- clean CV: at least **+0.3 points** and at least **10 of 15 folds** improved;
- confirmation on **new seeds (1, 2, 3)**, never used to choose: at least **+0.15 points**;
- permuted team keys: less than +0.2 points over the model without team features, and no more than 0.2 points below R1 + R3
  with the same permuted keys (a leak inside the training rows would make validation worse, not better).

This replaces an earlier version of the rule, changed before any run: the condition on teams never seen is now descriptive, because
the final set follows the test distribution. "Unseen both" also slightly favours B, because the partly revealed validation keys
stay in the pool. If no candidate passes, the encoder stays as it is.

## 0. Setup

In [1]:
import os
import sys
import json
import time
import warnings
from pathlib import Path

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features import team_key_features
from src.models import get_model_specs, with_params
from src.team_strength import KEY_P1, KEY_P2, TeamStrengthEncoder, unseen_cv_scores, with_team_strength
from src.validation import REPEAT_SEEDS, SEED, cv_scores

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
PROC = ROOT / "data" / "processed"
OUT = PROC / "models"
OUT.mkdir(parents=True, exist_ok=True)

team, states = [pd.read_pickle(PROC / f"{name}_train.pkl") for name in ("team", "states")]
ref = pd.read_pickle(PROC / "features_train.pkl")             # v1 features (notebook 03)
with open(PROC / "feature_groups.json") as f:
    groups = json.load(f)
y = ref.pop("player_won")
NUM = [c for c in ref.columns if c not in groups["species_cat"]]
KEYS = [KEY_P1, KEY_P2]
X = pd.concat([ref[NUM], team_key_features(team, states).reindex(ref.index)], axis=1)
assert X[KEYS].notna().all().all()
print(f"battles: {len(X):,} | features: {len(NUM)} + team keys")

specs = get_model_specs()
LGBM = with_params(specs["lightgbm"], {
    "model__colsample_bytree": 0.6286197472598067, "model__learning_rate": 0.006552992991573526,
    "model__min_child_samples": 106, "model__n_estimators": 1424, "model__num_leaves": 12,
    "model__reg_alpha": 0.6070874110546393, "model__reg_lambda": 2.7129089558470034,
    "model__subsample": 0.6629794526009424})

REFERENCE_NAME = "R1 + R3 (reference)"
REFERENCE = {"train_unseen": 0.3, "pairs": True, "per_side": True}    # encoder of submissions 27-29
ENCODERS = {
    REFERENCE_NAME: REFERENCE,
    "no mask": {"pairs": True},
    "B soft P2": {**REFERENCE, "soft_p2": True},
    "C residual": {**REFERENCE, "residual": True},
}
CANDIDATES = [name for name in ENCODERS if name != REFERENCE_NAME]
NEW_SEEDS = (1, 2, 3)

battles: 10,000 | features: 155 + team keys


## 1. Checks on the encoder

- **Output columns:** the reference must still produce the eight team columns of submissions 27-29, in the same order.
- **Own label:** the label of one battle at a time is flipped, on 12 battles, and that battle's own team columns are compared.
  They must not change at all for the reference, no mask and B, because each row is encoded from a pool without its label. With C the
  logistic regression sees all the training rows, so a small change is expected: it is reported and bounded.
- **Coverage and correlations:** how many battles B adds to the pool, and how the encoded columns correlate with the winner.

In [2]:
Xk = X[NUM + KEYS]
out = ["te_p1", "te_p2", "te_n_p1", "te_n_p2", "diff_te", "pr1", "pr2", "diff_pr"]
encodings = {name: TeamStrengthEncoder(**params).fit_transform(Xk, y) for name, params in ENCODERS.items()}
assert list(encodings[REFERENCE_NAME].columns[-len(out):]) == out

rng = np.random.default_rng(SEED)
positions = rng.choice(len(X), size=12, replace=False)
for name, params in ENCODERS.items():
    largest = 0.0
    for pos in positions:
        flipped = y.copy()
        flipped.iloc[pos] = 1 - flipped.iloc[pos]
        changed = TeamStrengthEncoder(**params).fit_transform(Xk, flipped)
        largest = max(largest, (encodings[name][out].iloc[pos] - changed[out].iloc[pos]).abs().max())
    print(f"{name:22s} largest change of a battle's own team columns after flipping its label: {largest:.2e}")
    if name == "C residual":
        assert largest < 0.01, "residual: a row's label moves its own encoding too much"
    else:
        assert largest == 0.0, f"{name}: a row sees its own label"

n_revealed = X[KEY_P2].str.count(r"\|") + 1
print("\nP2 species revealed:", n_revealed.value_counts().sort_index().to_dict())
print(f"battles in the pool through P2 today: {(n_revealed == 6).mean():.1%} | with soft_p2: {(n_revealed >= 3).mean():.1%}")
pd.DataFrame({name: {c: enc[c].corr(y) for c in ["te_p1", "te_p2", "diff_te", "diff_pr"]} for name, enc in encodings.items()}).T.round(4)

R1 + R3 (reference)    largest change of a battle's own team columns after flipping its label: 0.00e+00
no mask                largest change of a battle's own team columns after flipping its label: 0.00e+00
B soft P2              largest change of a battle's own team columns after flipping its label: 0.00e+00
C residual             largest change of a battle's own team columns after flipping its label: 1.56e-03

P2 species revealed: {1: 17, 2: 39, 3: 165, 4: 849, 5: 3373, 6: 5557}
battles in the pool through P2 today: 55.6% | with soft_p2: 99.4%


,te_p1,te_p2,diff_te,diff_pr
R1 + R3 (reference),0.1058,-0.0713,0.1245,0.1083
no mask,0.1090,-0.0848,0.1358,0.1083
B soft P2,0.1169,-0.0847,0.1417,0.1083
C residual,0.0979,-0.0665,0.1154,0.0808


## 2. Paired CV: clean and with teams never seen

Every configuration on the same 15 folds. `no team features` does not depend on the team pool at all. Teams never seen are
descriptive only, for the report.

In [3]:
def repeated(fn, seeds=REPEAT_SEEDS):
    return np.concatenate([fn(s) for s in seeds])

t0 = time.time()
no_team = repeated(lambda s: cv_scores(LGBM, X[NUM], y, seed=s))
scores = {}
for name, params in ENCODERS.items():
    model = with_team_strength(LGBM, **params)
    scores[name] = {
        "clean": repeated(lambda s: cv_scores(model, X[NUM + KEYS], y, seed=s)),
        "unseen P1": repeated(lambda s: unseen_cv_scores(model, X[NUM + KEYS], y, side="p1", seed=s)),
        "unseen both": repeated(lambda s: unseen_cv_scores(model, X[NUM + KEYS], y, side="both", seed=s)),
    }
    print(f"{name:22s} clean {scores[name]['clean'].mean():.4f} | unseen P1 {scores[name]['unseen P1'].mean():.4f} | "
          f"unseen both {scores[name]['unseen both'].mean():.4f} ({time.time() - t0:.0f}s)")
print(f"{'no team features':22s} {no_team.mean():.4f}")

R1 + R3 (reference)    clean 0.6888 | unseen P1 0.6788 | unseen both 0.6771 (54s)
no mask                clean 0.6898 | unseen P1 0.6748 | unseen both 0.6737 (93s)
B soft P2              clean 0.6870 | unseen P1 0.6767 | unseen both 0.6739 (135s)
C residual             clean 0.6883 | unseen P1 0.6782 | unseen both 0.6761 (174s)
no team features       0.6779


In [4]:
rows = []
for name in ENCODERS:
    row = {"encoder": name}
    for mode in ("clean", "unseen P1", "unseen both"):
        d_ref = scores[name][mode] - scores[REFERENCE_NAME][mode]
        row[f"{mode}: acc"] = scores[name][mode].mean()
        row[f"{mode}: vs reference"] = d_ref.mean() * 100
        row[f"{mode}: folds"] = int((d_ref > 0).sum())
        row[f"{mode}: vs no team"] = (scores[name][mode] - no_team).mean() * 100
    rows.append(row)
table = pd.DataFrame(rows).set_index("encoder")
table.to_csv(OUT / "encoder_refinements.csv")
table.T.round(4)

encoder,R1 + R3 (reference),no mask,B soft P2,C residual
clean: acc,0.6888,0.6898,0.6870,0.6883
clean: vs reference,0.0000,0.0967,-0.1767,-0.0533
clean: folds,0.0000,9.0000,4.0000,6.0000
clean: vs no team,1.0900,1.1867,0.9133,1.0367
unseen P1: acc,0.6788,0.6748,0.6767,0.6782
unseen P1: vs reference,0.0000,-0.3967,-0.2100,-0.0600
unseen P1: folds,0.0000,1.0000,4.0000,8.0000
unseen P1: vs no team,0.0867,-0.3100,-0.1233,0.0267
unseen both: acc,0.6771,0.6737,0.6739,0.6761
unseen both: vs reference,0.0000,-0.3400,-0.3233,-0.1033


## 3. Confirmation on new seeds

The candidate with the best clean CV is chosen above, so its gain there is slightly optimistic. Seeds 1, 2 and 3 were never used to
choose it.

In [5]:
best = max(CANDIDATES, key=lambda name: scores[name]["clean"].mean())
confirm_best = repeated(lambda s: cv_scores(with_team_strength(LGBM, **ENCODERS[best]), X[NUM + KEYS], y, seed=s), NEW_SEEDS)
confirm_ref = repeated(lambda s: cv_scores(with_team_strength(LGBM, **REFERENCE), X[NUM + KEYS], y, seed=s), NEW_SEEDS)
confirm_gain = (confirm_best - confirm_ref).mean() * 100
print(f"best candidate: {best} | new seeds {confirm_best.mean():.4f} vs reference {confirm_ref.mean():.4f} | "
      f"{confirm_gain:+.2f} points, better in {(confirm_best > confirm_ref).sum()}/15 folds")

best candidate: no mask | new seeds 0.6868 vs reference 0.6863 | +0.04 points, better in 7/15 folds


## 4. Control: permuted team keys

The same permutation for the reference and for every candidate. A gain over the model without team features would mean signal that
does not come from the real teams. A score far below the reference would point to a leak inside the training rows.

In [6]:
rng = np.random.default_rng(SEED)
X_perm = X.copy()
X_perm[KEYS] = X[KEYS].to_numpy()[rng.permutation(len(X))]
perm = {name: repeated(lambda s: cv_scores(with_team_strength(LGBM, **params), X_perm[NUM + KEYS], y, seed=s))
        for name, params in ENCODERS.items()}
perm_table = pd.DataFrame({name: {"permuted acc": p.mean(), "vs no team": (p - no_team).mean() * 100,
                                  "vs reference": (p - perm[REFERENCE_NAME]).mean() * 100} for name, p in perm.items()}).T
perm_table.round(4)

,permuted acc,vs no team,vs reference
R1 + R3 (reference),0.6750,-0.2933,0.0000
no mask,0.6753,-0.2567,0.0367
B soft P2,0.6740,-0.3867,-0.0933
C residual,0.6749,-0.3000,-0.0067


## 5. Decision

Rule fixed before running, for the candidate with the best clean CV: clean at least +0.3 points and at least 10/15 folds against
R1 + R3; new seeds at least +0.15 points; permuted keys below +0.2 points over no team features and no more than 0.2 points below the
reference.

In [7]:
checks = {
    "clean >= +0.3": table.loc[best, "clean: vs reference"] >= 0.3,
    "clean folds >= 10": table.loc[best, "clean: folds"] >= 10,
    "new seeds >= +0.15": confirm_gain >= 0.15,
    "permuted vs no team < +0.2": perm_table.loc[best, "vs no team"] < 0.2,
    "permuted vs reference >= -0.2": perm_table.loc[best, "vs reference"] >= -0.2,
}
print(f"best candidate: {best}")
for rule, ok in checks.items():
    print(f"{'OK' if ok else 'NO'} {rule}")
print(f"ADOPT {best}" if all(checks.values()) else "DO NOT ADOPT: the encoder stays as in submissions 27-29")

best candidate: no mask
NO clean >= +0.3
NO clean folds >= 10
NO new seeds >= +0.15
OK permuted vs no team < +0.2
OK permuted vs reference >= -0.2
DO NOT ADOPT: the encoder stays as in submissions 27-29


## 6. Summary

- **Checks:** flipping a battle's label leaves its own team columns unchanged for the reference, no mask and B; with C the largest
  change is 0.0016. With permuted keys every encoder stays below the model without team features (-0.26 to -0.39 points) and
  within 0.1 points of the reference.
- **Clean CV against R1 + R3 (0.6888):** no mask 0.6898 (+0.10, 9/15 folds), B 0.6870 (-0.18, 4/15), C 0.6883 (-0.05, 6/15).
- **New seeds, best candidate (no mask):** +0.04 points, 7/15 folds.
- **Teams never seen (descriptive), both sides, against R1 + R3:** no mask -0.34, B -0.32, C -0.10 points.
- B raises the correlation of `diff_te` with the winner on the training rows (0.1245 to 0.1417) without raising accuracy.

**Decision by the rule fixed before running: not adopted.** The encoder stays as in submissions 27-29.